# Exploratory Data Analysis & Time-Series Preprocessing
## NASA C-MAPSS Turbofan Degradation Dataset (FD001)
**Framework**: An Explainable and Uncertainty-Aware Deep Learning Framework for Predicting Remaining Useful Life  
**Target**: Academic Research Portfolio for German M.Sc. Artificial Intelligence Admissions


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path(".." if Path(".").resolve().name == "notebooks" else ".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import set_seed, get_device
from src.data_loader import CMAPSSDataLoader, SENSOR_COLS, SETTING_COLS
from src.preprocessing import CMAPSSPreprocessor
from src.dataset import build_dataloaders

set_seed(42)
device = get_device("mps")
print(f"Target compute acceleration device: {device}")

### 1. Ingesting Raw Run-to-Failure Sensor Trajectories
Loading subset **FD001** (1 operating condition, 1 fault mode: High-Pressure Compressor degradation).

In [ ]:
loader = CMAPSSDataLoader(data_dir=PROJECT_ROOT / "data" / "raw" / "CMAPSSData")
train_df = loader.load_train_df("FD001", max_rul=125)
test_df, test_rul = loader.load_test_df("FD001", max_rul=125)

print(f"Train DataFrame Shape: {train_df.shape} across {train_df['engine_id'].nunique()} engines")
print(f"Test DataFrame Shape:  {test_df.shape} across {test_df['engine_id'].nunique()} engines")
train_df.head()

### 2. Sensor Variance Profiling & Invariant Feature Pruning
In FD001, sensors operating at constant ambient sea-level conditions exhibit zero variance (std < 0.01). Removing them eliminates numerical instability and prevents zero-division during Z-score scaling.

In [ ]:
preprocessor = CMAPSSPreprocessor(include_settings=False, variance_threshold=0.01)
features = preprocessor.select_features(train_df)

print(f"Pruned Invariant Features: {preprocessor.constant_cols}")
print(f"Retained {len(features)} Informative Sensors: {features}")

### 3. Pearson Correlation Heatmap with Degradation Target (RUL)

In [ ]:
corr = train_df[features + ["target_rul"]].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Sensor Correlation Matrix with RUL (FD001)")
plt.show()

### 4. Engine-Level Splitting & Sliding Window Sequence Generation
Ensuring **zero temporal data leakage** by partitioning engines 80/20 at the engine ID level.

In [ ]:
train_part, val_part = preprocessor.split_train_val(train_df, val_ratio=0.20, seed=42)
preprocessor.fit(train_part)

train_norm = preprocessor.transform(train_part)
val_norm = preprocessor.transform(val_part)
test_norm = preprocessor.transform(test_df)

X_train, y_train, ids_train = preprocessor.create_sliding_windows(train_norm)
X_val, y_val, ids_val = preprocessor.create_sliding_windows(val_norm)
X_test, y_test, ids_test = preprocessor.create_test_windows(test_norm, last_cycle_only=True)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}, y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")

loaders = build_dataloaders(X_train, y_train, X_val, y_val, X_test, y_test, batch_size=32)
batch_x, batch_y, batch_id = next(iter(loaders["train"]))
print(f"Batch X Tensor: {batch_x.shape}, Batch y Tensor: {batch_y.shape}")